## Q3 — Missing Driver Code Logic
Inspecting `drivers.csv` revealed that some drivers (mostly historical, pre-1980s) 
have no official FIA three-letter code. These appear as NULL, empty string, or the 
literal string "\N" (a common CSV null marker). For these drivers, we derive a code 
by taking the first 3 letters of their surname and converting to uppercase 
(e.g., Fangio → FAN). This matches the convention F1 uses for drivers without an 
assigned code.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_races     = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",     header=True)
df_results   = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv",   header=True)
df_drivers   = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv",   header=True)

In [0]:
#Parse duration → seconds
df_pit_stops = df_pit_stops.withColumn(
    "duration_sec",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":").getItem(0).cast("double") * 60 +
        F.split(F.col("duration"), ":").getItem(1).cast("double")
    ).otherwise(
        F.col("duration").cast("double")
    )
)

In [0]:
df_driver_avg = (
    df_pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.round(F.avg("duration_sec"), 3).alias("avg_pit_stop_sec"))
)

In [0]:
df_results_clean = df_results.select(
    "raceId", "driverId",
    F.col("positionOrder").cast("int").alias("finishing_position")
)

In [0]:

# Fill missing code with first 3 letters of surname
df_drivers_clean = df_drivers.select(
    "driverId",
    F.when(
        F.col("code").isNull() | (F.trim(F.col("code")) == ""),
        F.upper(F.substring(F.col("surname"), 1, 3))
    ).otherwise(
        F.col("code")
    ).alias("driver_code"),
    F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("driver_name")
)

In [0]:

# ── 6. Check how many codes were missing and got filled ───────────────────────
missing_count = df_drivers.filter(
    F.col("code").isNull() | (F.trim(F.col("code")) == "")
).count()
print(f"Drivers with missing code (filled by surname): {missing_count}")

In [0]:

df_joined = (
    df_driver_avg
    .join(df_results_clean,                                    on=["raceId", "driverId"], how="left")
    .join(df_races.select("raceId", "year", F.col("name").alias("race_name")), on="raceId", how="left")
    .join(df_drivers_clean,                                    on="driverId",             how="left")
)

In [0]:
#Rank by finishing position within each race 
window = Window.partitionBy("raceId").orderBy("finishing_position")

df_q3 = (
    df_joined
    .withColumn("rank", F.rank().over(window))
    .select("year", "race_name", "finishing_position", "rank",
            "driver_code", "driver_name", "avg_pit_stop_sec")
    .orderBy("year", "race_name", "finishing_position")
)


In [0]:
display(df_q3)